# Exploring the results

This notebook reads only what is already in `results/` — it runs no models and
needs no GPU. Use it to poke at the raw scores behind the paper's claims, or to
check a number the manuscript reports.

Sign convention, used everywhere:

```
G(a) = log P(a | before the attempt) - log P(a | after it failed)
```

`G > 0` means the failure record made the failed action less likely, which is
what a harness is trying to achieve. `G < 0` is feedback inversion.

In [ ]:
import glob, os, sys
sys.path.insert(0, os.path.join('..', 'src'))
import numpy as np, pandas as pd

from slmecho.analysis import load_raw, to_wide, derive, item_metadata, summarise, attach_model_info

ROOT = '..'
paths = [p for p in glob.glob(os.path.join(ROOT, 'results/raw/probe/*__*.jsonl'))
         if not p.endswith('.greedy.jsonl')]
raw = load_raw(paths)
meta = item_metadata({'toolshed': os.path.join(ROOT, 'data/probe_toolshed.jsonl'),
                      'coderepair': os.path.join(ROOT, 'data/probe_coderepair.jsonl')})
print(f'{len(raw):,} scored (item, condition, candidate) triples')
raw.head()

## From raw scores to the paper's quantities

`derive` adds every measure the paper reports. The identity `-G = copy + sem`
holds exactly by construction, which the assertion below checks.

In [ ]:
d = derive(to_wide(raw))
ok = d.dropna(subset=['G', 'copy', 'sem'])
assert np.allclose(-ok['G'], ok['copy'] + ok['sem']), 'decomposition identity broken'
print('decomposition identity holds on', len(ok), 'items')
d.groupby(['model', 'env'])[['G', 'copy', 'sem', 'G_per_token']].mean().round(2)

## The item-level view

Averages can hide a handful of extreme items. The distribution below is the
reason the paper reports the *fraction of items* with a negative gain: the
effect is not carried by outliers.

In [ ]:
import matplotlib.pyplot as plt
from slmecho.plotting import use_paper_style, BLUE, ORANGE
use_paper_style()

sub = d[(d['env'] == 'toolshed')]
fig, ax = plt.subplots(figsize=(5, 2.6))
for i, (m, g) in enumerate(sub.groupby('model')):
    ax.hist(g['G'].dropna(), bins=30, histtype='step', linewidth=1.4, label=m)
ax.axvline(0, color='#9a9a9a', linestyle=(0, (4, 3)), linewidth=0.9)
ax.set_xlabel('corrective gain $G$ (nats)'); ax.set_ylabel('items')
ax.legend(); plt.show()

print((sub.groupby('model')['G'].apply(lambda s: (s < 0).mean())).round(3))

## Which errors are worst?

If the effect were an artefact of one error message's wording, it would show up
as one family dominating this table.

In [ ]:
m = d.merge(meta, on=['env', 'item_id'], how='left')
(m[m.env == 'toolshed']
   .pivot_table(index='error_type', columns='model', values='G', aggfunc='mean')
   .round(1)
   .sort_values(by=m['model'].dropna().unique()[0] if len(m['model'].dropna()) else 'error_type'))

## Reading actual trajectories

The rollout logs contain every action and observation. This is the quickest way
to see the phenomenon rather than measure it.

In [ ]:
import json
files = sorted(glob.glob(os.path.join(ROOT, 'results/raw/agent/*.jsonl')))
print('\n'.join(os.path.basename(f) for f in files))
if files:
    rows = [json.loads(l) for l in open(files[0], encoding='utf-8')]
    worst = max(rows, key=lambda r: r.get('n_exact_repeats', 0))
    print(f"\n{worst['task_id']}  solved={worst['solved']}  "
          f"exact repeats={worst['n_exact_repeats']}")
    for i, (a, ok) in enumerate(zip(worst['actions'], worst['ok']), 1):
        print(f"  {i}. [{'ok ' if ok else 'ERR'}] {a[:100]}")